# 使用 1.5 端口 NanoVNA V2 测量 4 端口设备

NanoVNA V2 是一款 1.5 端口 VNA，这意味着它只能通过其两个端口测量正向反射和传输系数 $S_{11}$ 和 $S_{21}$。这被称为一种三接收器 VNA 架构，尽管存在此限制，但仍然可以进行完全校准的二端口测量。我们建议读者首先阅读 [使用三个接收器进行校准](./Calibration With Three Receivers.ipynb)，以便熟悉基本原理。简而言之，要测量 2 端口器件的逆向系数 $S_{12}$ 和 $S_{22}$，需要再次测量，并对端口方向进行物理翻转（或模拟翻转），如 [此示例](TwoPortOnePath, EnhancedResponse, and FakeFlip.ipynb) 中所述。如果被测器件 (DUT) 具有更多的端口，问题会变得更加严重，如 [此示例](./Measuring a Mutiport Device with a 2-Port Network Analyzer.ipynb) 中所述。在本示例中，使用 NanoVNA V2 和匹配端口技术测量了一个 [4 端口 SMA 分路器](https://www.minicircuits.com/WebStore/dashboard.html?model=ZX10Q-2-19-S%2B)。它涉及测量所有端口组合的 4 端口器件（12 次测量），并将未使用的端口端接为 50 欧姆匹配。对于 VNA 校准，还需要进行四次测量，使用 SMA 校准标准（短路/开路/负载/直通）。DUT 的制造商在其网站上提供了测量的散射参数，稍后将用于比较。

## 数据采集

要使用 scikit-rf 的双端口校准类，例如 `TwoPortOnePath`，需要提供 DUT（待测设备）和校准标准的各个测量结果，这些结果应为包含 $S_{11}$ 和 $S_{21}$ 数据的双端口网络。以下代码片段可用于配置 NanoVNA，并获取和保存数据：

```pythonimport skrffrom skrf.vi.vna.nanovna import NanoVNAv2# 连接到 /dev/ttyACM0 上的 NanoVNA（Linux）nanovna = NanoVNAv2('ASRL/dev/ttyACM0::INSTR')# 对于 Windows 用户：COM1 对应 ASRL1# nanovna = NanoVNAv2('ASRL1::INSTR')# 配置频率扫描（例如，从 1 MHz 到 4.4 GHz，步长为 1 MHz）f_start = 1e6f_stop = 4.4e9f_step = 1e6num = int(1 + (f_stop - f_start) / f_step)nanovna.frequency = skrf.Frequency(start=f_start, stop=f_stop, npoints=num)# 测量 4 端口的所有 12 种组合n_ports = 4for i_src in range(n_ports):    for i_sink in range(n_ports):        if i_sink != i_src:            input('连接 vna_p1 -> dut_p{}, vna_p2 -> dut_p{}，然后按 ENTER 键：'.format(i_src + 1, i_sink + 1))            nw_raw = nanovna.get_snp_network(ports=(1, 2))            nw_raw.write_touchstone('./data_MiniCircuits_splitter/dut_raw_{}{}'.format(i_sink + 1, i_src + 1))```

校准标准应使用相同的重复调用 `get_snp_network(ports=(1, 2))` 和 `write_touchstone()` 来进行测量。

## 使用 `TwoPortOnePath` 进行离线校准

通过 USB 从 NanoVNA 传输的测量数据始终是原始数据（未经校准），无论 NanoVNA 本身是否进行了任何校准。因此，需要使用离线校准来校正数据。将校准标准的测量结果存储为单独的二端口网络，可以使用 scikit-rf 轻松创建 TwoPortOnePath 校准。在此示例中，假设测量的短路、开路、匹配和直通的阻抗和相位延迟是理想的，即没有任何损耗或偏移：

In [ ]:
import skrf
from skrf.calibration import TwoPortOnePath

# load networks of the raw calibration standard measurements
short_raw = skrf.Network('./data_MiniCircuits_splitter/cal_short_raw.s2p')
open_raw = skrf.Network('./data_MiniCircuits_splitter/cal_open_raw.s2p')
match_raw = skrf.Network('./data_MiniCircuits_splitter/cal_match_raw.s2p')
thru_raw = skrf.Network('./data_MiniCircuits_splitter/cal_thru_raw.s2p')

# create an ideal 50-Ohm line for the short, open, match and through reference responses ("ideals")
line = skrf.media.DefinedGammaZ0(frequency=short_raw.frequency, z0=50)

# create and run the calibration
cal = TwoPortOnePath(ideals=[line.short(nports=2), line.open(nports=2), line.match(nports=2), line.thru()],
                     measured=[short_raw, open_raw, match_raw, thru_raw],
                     n_thrus=1, source_port=1)
cal.run()

现在可以使用此校准来校正 12 个独立的 2 端口子网络。为了进行完全校正，需要成对地提供具有正向和反向测量的子网络，例如将 ($S_{32}$, $S_{22}$) 与 ($S_{23}$, $S_{33}$) 配对。可以使用嵌套循环来处理这一点。为了与制造商提供的测量结果进行比较，将校准后的结果存储在一个 4 端口网络中会很方便，然后可以轻松地绘制图形：

In [ ]:
import numpy as np

# create an empty array (f x 4 x 4) for the 4-port to be filled
s = np.zeros((len(short_raw.frequency), 4, 4), dtype=complex)
splitter_cal = skrf.Network(frequency=short_raw.frequency, s=s)

# loop through all 12 measurements, apply the calibration and save it inside 4-port network
for i_src in range(4):
    for i_recv in range(4):
        if i_src != i_recv:
            dut_raw_fwd = skrf.Network(f'./data_MiniCircuits_splitter/dut_raw_{i_recv + 1}{i_src + 1}.s2p')
            dut_raw_rev = skrf.Network(f'./data_MiniCircuits_splitter/dut_raw_{i_src + 1}{i_recv + 1}.s2p')
            dut_cal = cal.apply_cal((dut_raw_fwd, dut_raw_rev))

            # dut_cal is now a fully populated and corrected 2-port; save it in splitter_cal
            splitter_cal.s[:, i_src, i_src] = dut_cal.s[:, 0, 0]
            splitter_cal.s[:, i_recv, i_src] = dut_cal.s[:, 1, 0]
            splitter_cal.s[:, i_src, i_recv] = dut_cal.s[:, 0, 1]
            splitter_cal.s[:, i_recv, i_recv] = dut_cal.s[:, 1, 1]

结果已进行校正并合并为一个四端口网络。为了进行比较，将幅值与制造商提供的测量结果一起绘制：

In [ ]:
import matplotlib.pyplot as mplt

# load reference results by MiniCircuits
splitter_mc = skrf.Network('./data_MiniCircuits_splitter/MiniCircuits_ZX10Q-2-19-S+___Plus25degC.s4p')

# plot both results
fig, ax = mplt.subplots(4, 4)
fig.set_size_inches(12, 8)

for i in range(4):
    for j in range(4):
        splitter_cal.plot_s_db(i, j, ax=ax[i][j])
        splitter_mc.plot_s_db(i, j, ax=ax[i][j])
        ax[i][j].get_legend().remove()
        ax[i][j].set_xlim(0, 4.4e9)
fig.legend(['NanoVNA_cal', 'Manufacturer'], loc='upper center', ncol=2)
fig.tight_layout(rect=(0, 0, 1, 0.95))
mplt.show()

校准后的结果与制造商提供的数据非常接近。他们显然使用了 Keysight N5242A PNA-X（26.5 GHz 4 端口 VNA）进行测量。考虑到 200 美元的 NanoVNA V2 以及针对匹配端口技术所做的各种假设（DUT 未使用的端口的端接肯定并非完全没有反射），这些结果已经相当不错了。